In [14]:
# Step 1: Import necessary libraries
import zipfile
import os
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, accuracy_score


# Step 2: Define path to ZIP file and extract
zip_path = '/content/archive (2).zip'  # Replace this if path is different
extract_dir = '/content/data_extracted'

with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

# Step 3: Load data directly from extracted files
train_path = os.path.join(extract_dir, 'Training.csv')
test_path = os.path.join(extract_dir, 'Testing.csv')

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

# Drop the 'Unnamed: 133' column if it exists in train_df
if 'Unnamed: 133' in train_df.columns:
    train_df = train_df.drop('Unnamed: 133', axis=1)

# Step 4: Preprocess - Encode labels
label_encoder = LabelEncoder()
train_df['prognosis'] = label_encoder.fit_transform(train_df['prognosis'])
test_df['prognosis'] = label_encoder.transform(test_df['prognosis'])

X_train = train_df.drop('prognosis', axis=1)
y_train = train_df['prognosis']

X_test = test_df.drop('prognosis', axis=1)
y_test = test_df['prognosis']

# Step 5: Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Step 6: Train model
clf = RandomForestClassifier(random_state=42)
clf.fit(X_train_scaled, y_train)

# Step 7: Evaluate model
y_pred = clf.predict(X_test_scaled)
print("Accuracy:", accuracy_score(y_test, y_pred))
print("\nClassification Report:\n", classification_report(y_test, y_pred, target_names=label_encoder.classes_))

Accuracy: 0.9761904761904762

Classification Report:
                                          precision    recall  f1-score   support

(vertigo) Paroymsal  Positional Vertigo       1.00      1.00      1.00         1
                                   AIDS       1.00      1.00      1.00         1
                                   Acne       1.00      1.00      1.00         1
                    Alcoholic hepatitis       1.00      1.00      1.00         1
                                Allergy       1.00      1.00      1.00         1
                              Arthritis       1.00      1.00      1.00         1
                       Bronchial Asthma       1.00      1.00      1.00         1
                   Cervical spondylosis       1.00      1.00      1.00         1
                            Chicken pox       1.00      1.00      1.00         1
                    Chronic cholestasis       1.00      1.00      1.00         1
                            Common Cold       1.00    

In [15]:
# Step 8: Predict disease from single input
def predict_disease(symptom_dict):
    # Create a DataFrame with one row and all features
    input_df = pd.DataFrame([symptom_dict], columns=X_train.columns)

    # Fill missing symptoms with 0 (assume not present)
    input_df = input_df.fillna(0)

    # Scale the input using the same scaler
    input_scaled = scaler.transform(input_df)

    # Predict using the trained model
    prediction = clf.predict(input_scaled)
    disease = label_encoder.inverse_transform(prediction)[0]

    return disease


# Example: Create a symptom dictionary where symptoms are set to 1 (present), 0 (absent)
# You can replace this with actual user input in a real app
user_input = {
    col: 0 for col in X_train.columns
}
user_input["fatigue"] = 1
user_input["high_fever"] = 1
user_input["chills"] = 1
user_input["cough"] = 1

# Make prediction
predicted_disease = predict_disease(user_input)
print("Predicted Disease:", predicted_disease)


Predicted Disease: Bronchial Asthma
